In [1]:
import os
import glob
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, concatenate, add, GlobalAveragePooling2D, Dense, Reshape, multiply, MultiHeadAttention, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

IMG_WIDTH = 256
IMG_HEIGHT = 256
SEED_VALUE = 42
EPOCHS = 30  
BATCH_SIZE = 8

2026-06-24 14:16:26.806355: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782310587.061265      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782310587.137994      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782310587.738655      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782310587.738721      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782310587.738724      23 computation_placer.cc:177] computation placer alr

In [2]:
base_input_path_cvc = "/kaggle/input/datasets/quanhh42/multiresunet-datasets/CVC-ClinicDB/CVC-ClinicDB"

def load_cvc_clinicdb_data_full():
    all_img_dirs = glob.glob(os.path.join(base_input_path_cvc, '**', '*Original*'), recursive=True)
    all_msk_dirs = glob.glob(os.path.join(base_input_path_cvc, '**', '*Ground Truth*'), recursive=True)
    all_img_dirs = [d for d in all_img_dirs if os.path.isdir(d)]
    all_msk_dirs = [d for d in all_msk_dirs if os.path.isdir(d)]
    image_dict = {}
    for img_dir in all_img_dirs:
        for img_path in glob.glob(os.path.join(img_dir, '*.*')):
            file_stem = os.path.splitext(os.path.basename(img_path))[0] 
            if file_stem not in image_dict:
                image_dict[file_stem] = img_path
    mask_dict = {}
    for msk_dir in all_msk_dirs:
        for msk_path in glob.glob(os.path.join(msk_dir, '*.*')):
            file_stem = os.path.splitext(os.path.basename(msk_path))[0]
            if file_stem not in mask_dict:
                mask_dict[file_stem] = msk_path
    common_stems = sorted(list(set(image_dict.keys()).intersection(set(mask_dict.keys()))))
    if len(common_stems) == 0:
        return np.random.rand(50, IMG_HEIGHT, IMG_WIDTH, 3).astype(np.float32), np.random.randint(0, 2, (50, IMG_HEIGHT, IMG_WIDTH, 1)).astype(np.float32)
    X, Y = [], []
    for stem in tqdm(common_stems, desc="Processing CVC Images"):
        img_path = image_dict[stem]
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) 
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT)) / 255.0
        X.append(img)
        msk_path = mask_dict[stem]
        msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        msk = cv2.resize(msk, (IMG_WIDTH, IMG_HEIGHT)) / 255.0
        Y.append(np.round(msk, 0))
    return np.array(X, dtype=np.float32), np.expand_dims(np.array(Y, dtype=np.float32), -1)

X_cvc, Y_cvc = load_cvc_clinicdb_data_full()
X_train, X_val, y_train, y_val = train_test_split(X_cvc, Y_cvc, test_size=0.2, random_state=SEED_VALUE)

Processing CVC Images: 100%|██████████| 612/612 [00:18<00:00, 33.88it/s]


In [3]:
def multi_res_block(u, num_filters, alpha=1.67):
    w = num_filters * alpha
    k1 = int(w * 0.167)
    k2 = int(w * 0.333)
    k3 = int(w * 0.5)
    shortcut = u
    shortcut = Conv2D(k1 + k2 + k3, (1, 1), padding='same', use_bias=False)(shortcut)
    shortcut = BatchNormalization()(shortcut)
    conv3x3 = Conv2D(k1, (3, 3), padding='same', activation='relu')(u)
    conv3x3 = BatchNormalization()(conv3x3)
    conv5x5 = Conv2D(k2, (3, 3), padding='same', activation='relu')(conv3x3)
    conv5x5 = BatchNormalization()(conv5x5)
    conv7x7 = Conv2D(k3, (3, 3), padding='same', activation='relu')(conv5x5)
    conv7x7 = BatchNormalization()(conv7x7)
    out = concatenate([conv3x3, conv5x5, conv7x7], axis=-1)
    out = add([shortcut, out])
    out = Activation('relu')(out)
    return out

def se_block(u, ratio=8):
    channel = u.shape[-1]
    se = GlobalAveragePooling2D()(u)
    se = Dense(channel // ratio, activation='relu', use_bias=False)(se)
    se = Dense(channel, activation='sigmoid', use_bias=False)(se)
    se = Reshape((1, 1, channel))(se)
    return multiply([u, se])

def res_path(u, num_blocks, num_filters):
    for _ in range(num_blocks):
        shortcut = u
        shortcut = Conv2D(num_filters, (1, 1), padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)
        conv = Conv2D(num_filters, (3, 3), padding='same', activation='relu')(u)
        conv = BatchNormalization()(conv)
        u = add([shortcut, conv])
        u = Activation('relu')(u)
    return u

def mhsa_bottleneck(u, num_heads=4):
    shape = u.shape
    h, w, c = shape[1], shape[2], shape[3]
    flat = Reshape((h * w, c))(u)
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=c)(flat, flat)
    attn = add([flat, attn])
    attn = LayerNormalization()(attn)
    out = Reshape((h, w, c))(attn)
    return out

def hts_multiresunet(input_size=(256, 256, 3)):
    inputs = Input(input_size)
    mres1 = multi_res_block(inputs, 32)
    se1 = se_block(mres1)
    pool1 = MaxPool2D(pool_size=(2, 2))(se1)
    mres2 = multi_res_block(pool1, 64)
    se2 = se_block(mres2)
    pool2 = MaxPool2D(pool_size=(2, 2))(se2)
    mres3 = multi_res_block(pool2, 128)
    se3 = se_block(mres3)
    pool3 = MaxPool2D(pool_size=(2, 2))(se3)
    mres4 = multi_res_block(pool3, 256)
    se4 = se_block(mres4)
    pool4 = MaxPool2D(pool_size=(2, 2))(se4)
    b1 = multi_res_block(pool4, 512)
    b2 = mhsa_bottleneck(b1)
    res1 = res_path(se1, 4, 32)
    res2 = res_path(se2, 3, 64)
    res3 = res_path(se3, 2, 128)
    res4 = res_path(se4, 1, 256)
    up4 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(b2)
    merge4 = concatenate([up4, res4], axis=-1)
    dmres4 = multi_res_block(merge4, 256)
    dse4 = se_block(dmres4)
    up3 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(dse4)
    merge3 = concatenate([up3, res3], axis=-1)
    dmres3 = multi_res_block(merge3, 128)
    dse3 = se_block(dmres3)
    up2 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(dse3)
    merge2 = concatenate([up2, res2], axis=-1)
    dmres2 = multi_res_block(merge2, 64)
    dse2 = se_block(dmres2)
    up1 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(dse2)
    merge1 = concatenate([up1, res1], axis=-1)
    dmres1 = multi_res_block(merge1, 32)
    dse1 = se_block(dmres1)
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(dse1)
    return Model(inputs=inputs, outputs=outputs)

In [4]:
def get_edge_aware_focal_tversky_loss(alpha=0.3, beta=0.7, gamma=1.33, lamda=0.1):
    def edge_aware_focal_tversky(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        epsilon = 1e-6
        tp = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
        fn = tf.reduce_sum(y_true * (1.0 - y_pred), axis=[1, 2, 3])
        fp = tf.reduce_sum((1.0 - y_true) * y_pred, axis=[1, 2, 3])
        tversky_index = (tp + epsilon) / (tp + alpha * fn + beta * fp + epsilon)
        focal_tversky_loss = tf.pow((1.0 - tversky_index), gamma)
        focal_tversky_loss = tf.reduce_mean(focal_tversky_loss)
        y_true_edges = tf.image.sobel_edges(y_true) 
        y_pred_edges = tf.image.sobel_edges(y_pred)
        edge_loss = tf.reduce_mean(tf.square(y_true_edges - y_pred_edges))
        return focal_tversky_loss + (lamda * edge_loss)
    return edge_aware_focal_tversky

def jaccard_index(y_true, y_pred):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(tf.cast(y_pred, tf.float32))
    intersection = K.sum(y_true_f * y_pred_f)
    return (intersection + 1e-6) / (K.sum(y_true_f) + K.sum(y_pred_f) - intersection + 1e-6)

def dice_coefficient(y_true, y_pred):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(tf.cast(y_pred, tf.float32))
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + 1e-6) / (K.sum(y_true_f) + K.sum(y_pred_f) + 1e-6)

In [5]:
tuning_space = [
    {"alpha": 0.1, "beta": 0.9, "gamma": 1.33, "lamda": 0.1, "tag": "Alpha_0.1_Beta_0.9"},
    {"alpha": 0.3, "beta": 0.7, "gamma": 1.33, "lamda": 0.1, "tag": "Alpha_0.3_Beta_0.7_Proposed"},
    {"alpha": 0.5, "beta": 0.5, "gamma": 1.33, "lamda": 0.1, "tag": "Alpha_0.5_Beta_0.5"},
    {"alpha": 0.7, "beta": 0.3, "gamma": 1.33, "lamda": 0.1, "tag": "Alpha_0.7_Beta_0.3"},
    {"alpha": 0.3, "beta": 0.7, "gamma": 1.33, "lamda": 0.0, "tag": "Lamda_0.00"},
    {"alpha": 0.3, "beta": 0.7, "gamma": 1.33, "lamda": 0.05, "tag": "Lamda_0.05"},
    {"alpha": 0.3, "beta": 0.7, "gamma": 1.33, "lamda": 0.3, "tag": "Lamda_0.30"},
    {"alpha": 0.3, "beta": 0.7, "gamma": 1.33, "lamda": 0.5, "tag": "Lamda_0.50"},
    {"alpha": 0.3, "beta": 0.7, "gamma": 1.33, "lamda": 1.0, "tag": "Lamda_1.00"}
]

tuning_results = []

for idx, config in enumerate(tuning_space):
    print(f"\n[RUNNING] CVC-ClinicDB - Configuration {idx+1}/{len(tuning_space)}: {config['tag']}")
    K.clear_session()
    tf.compat.v1.reset_default_graph()
    
    model = hts_multiresunet(input_size=(IMG_HEIGHT, IMG_WIDTH, 3)) 
    
    custom_loss = get_edge_aware_focal_tversky_loss(
        alpha=config['alpha'], beta=config['beta'], gamma=config['gamma'], lamda=config['lamda']
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=custom_loss,
        metrics=[jaccard_index, dice_coefficient]
    )
    callbacks = [EarlyStopping(monitor='val_jaccard_index', patience=8, mode='max', restore_best_weights=True)]
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=0
    )
    eval_metrics = model.evaluate(X_val, y_val, verbose=0)
    val_jaccard = eval_metrics[1] * 100
    val_dice = eval_metrics[2] * 100
    
    print(f"--> [RESULT] CVC-ClinicDB | {config['tag']} -> Jaccard: {val_jaccard:.2f}% | Dice: {val_dice:.2f}%")
    
    tuning_results.append({
        "Dataset": "CVC-ClinicDB",
        "Configuration": config['tag'],
        "Alpha": config['alpha'],
        "Beta": config['beta'],
        "Gamma": config['gamma'],
        "Lamda": config['lamda'],
        "Jaccard (%)": round(val_jaccard, 2),
        "Dice (%)": round(val_dice, 2)
    })

df_results = pd.DataFrame(tuning_results)
df_results.to_csv("cvc_sensitivity_analysis.csv", index=False)
print(" FINAL PARAMETRIC SENSITIVITY MATRIX FOR CVC-ClinicDB")
print(df_results.to_string(index=False))


[RUNNING] CVC-ClinicDB - Configuration 1/9: Alpha_0.1_Beta_0.9


I0000 00:00:1782310644.806465      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1782310685.798482      67 service.cc:152] XLA service 0x7f7148002690 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782310685.798526      67 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1782310692.942122      67 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1782310741.800537      67 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-06-24 14:25:20.478501: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight System

--> [RESULT] CVC-ClinicDB | Alpha_0.1_Beta_0.9 -> Jaccard: 38.71% | Dice: 55.64%

[RUNNING] CVC-ClinicDB - Configuration 2/9: Alpha_0.3_Beta_0.7_Proposed
--> [RESULT] CVC-ClinicDB | Alpha_0.3_Beta_0.7_Proposed -> Jaccard: 65.30% | Dice: 78.91%

[RUNNING] CVC-ClinicDB - Configuration 3/9: Alpha_0.5_Beta_0.5
--> [RESULT] CVC-ClinicDB | Alpha_0.5_Beta_0.5 -> Jaccard: 57.53% | Dice: 72.87%

[RUNNING] CVC-ClinicDB - Configuration 4/9: Alpha_0.7_Beta_0.3
--> [RESULT] CVC-ClinicDB | Alpha_0.7_Beta_0.3 -> Jaccard: 65.58% | Dice: 79.19%

[RUNNING] CVC-ClinicDB - Configuration 5/9: Lamda_0.00
--> [RESULT] CVC-ClinicDB | Lamda_0.00 -> Jaccard: 62.79% | Dice: 77.12%

[RUNNING] CVC-ClinicDB - Configuration 6/9: Lamda_0.05
--> [RESULT] CVC-ClinicDB | Lamda_0.05 -> Jaccard: 56.73% | Dice: 72.34%

[RUNNING] CVC-ClinicDB - Configuration 7/9: Lamda_0.30
--> [RESULT] CVC-ClinicDB | Lamda_0.30 -> Jaccard: 63.15% | Dice: 77.33%

[RUNNING] CVC-ClinicDB - Configuration 8/9: Lamda_0.50
--> [RESULT] CVC-Clinic